# Analyse des défauts manqués (étape 6 de la phase finale)

**Question** : les défauts qu'**aucun modèle** ne détecte, qui sont-ils ? Deux issues possibles, comme le ferait un analyste de risque :
1. **diagnostic** : s'ils ressemblent à de très bons clients, la cible est en partie bruitée (la définition exacte de `dpnm` n'est pas documentée ; des défauts sans dette ou presque ont été vus dans l'EDA). C'est une limite à démontrer et à documenter (méthodologie, Streamlit) ;
2. **source de colonnes** : s'ils partagent un comportement que les colonnes actuelles ne décrivent pas, de nouvelles colonnes peuvent être construites. Chacune doit être **justifiée par la logique métier**, signalée dans le journal comme issue de cette analyse, puis jugée **sur les 75 % restants du train** (référence relancée sur cette même partie), et enfin par le test, lu une seule fois.

**Ce n'est pas un scénario** : aucun modèle n'est entraîné, rien n'est écrit dans le fichier de résultats, le test n'est pas lu.

**Protocole, fixé avant l'analyse (06/10, rappelé ici)** :
- **découpage du train fixé avant de regarder** : 25 % pour l'analyse, 75 % pour la validation ; tirage stratifié sur `dpnm`, `random_state=42`. Il est enregistré dans `data/ML/decoupage_analyse_defauts.csv` au premier lancement, puis relu et vérifié à chaque lancement : **il n'est jamais refait** ;
- **tout ce notebook ne regarde que la partie d'analyse** : la cible des 75 % de validation n'est jamais lue ici ;
- **défauts manqués** : défauts de la partie d'analyse qu'**aucun des modèles** de la version retenue ne signale, chacun à son seuil de fonctionnement (celui qui détecte 60 % des défauts, le rappel minimal des scénarios), calculé sur ses probabilités hors pli enregistrées (chaque client y est prédit par un modèle qui ne l'a pas vu) ;
- **quatre groupes**, dans la partie d'analyse : défauts manqués, défauts détectés (signalés par au moins un modèle), fausses alertes (bons clients signalés par au moins un modèle), bons clients non signalés.

In [1]:
# ==============================================================================
# RÉGLAGES DU LANCEMENT
# ==============================================================================
# Version retenue dont les probabilités hors pli sont lues ("14" ou "15", selon le duel ; puis la version finale)
BASE = "14"
VERSION_METHODE = "v8"
RAPPEL_MINIMAL = 0.60
PART_ANALYSE = 0.25   # fixé le 06/10, jamais revu d'après un résultat

## 1. Découpage fixé : partie d'analyse et partie de validation

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())
train = pd.read_csv(root_dir / "data" / "ML" / "train_dataset.csv")

# Découpage : tirage stratifié sur la cible, graine fixe ; enregistré au premier lancement, puis relu et vérifié
ids_analyse, _ = train_test_split(train["ID"], train_size=PART_ANALYSE, stratify=train["dpnm"], random_state=42)
decoupage = pd.DataFrame({"ID": train["ID"], "partie": np.where(train["ID"].isin(ids_analyse), "analyse", "validation")})
chemin = root_dir / "data" / "ML" / "decoupage_analyse_defauts.csv"
if chemin.exists():
    enregistre = pd.read_csv(chemin)
    assert enregistre.set_index("ID").loc[decoupage["ID"], "partie"].tolist() == decoupage["partie"].tolist(), \
        "Le découpage enregistré diffère du tirage : il ne doit jamais être refait"
    display(Markdown(f"Découpage relu dans `{chemin.name}` et vérifié (identique au tirage)."))
else:
    decoupage.to_csv(chemin, index=False)
    display(Markdown(f"Découpage enregistré dans `{chemin.name}` (premier lancement)."))

analyse = train[decoupage["partie"].to_numpy() == "analyse"].reset_index(drop=True)
display(Markdown(f"**Partie d'analyse** : {len(analyse)} clients sur {len(train)} ({len(analyse) / len(train):.0%}), "
                 f"dont {analyse['dpnm'].sum()} défauts. La partie de validation ({len(train) - len(analyse)} clients) n'est pas regardée ici."))

Découpage enregistré dans `decoupage_analyse_defauts.csv` (premier lancement).

**Partie d'analyse** : 4839 clients sur 19358 (25%), dont 771 défauts. La partie de validation (14519 clients) n'est pas regardée ici.

## 2. Les quatre groupes, à partir des probabilités hors pli de la version retenue

Pour chaque modèle de la version retenue, le seuil est celui qui détecte 60 % des défauts sur l'ensemble de ses probabilités hors pli (son point de fonctionnement dans le scénario). Un client est **signalé** si au moins un modèle le signale. Les tableaux ne portent que sur la partie d'analyse.

In [3]:
probas = pd.read_csv(root_dir / "data" / "ML" / "probas_hors_pli" / f"ml_{BASE}_{VERSION_METHODE}.csv").set_index("ID")
assert set(train["ID"]) == set(probas.index), f"Les probabilités de ml_{BASE} ne couvrent pas le train"
MODELES = [c for c in probas.columns if c not in ("dpnm", "version_methode")]

def seuil_pour_rappel(y, proba, rappel=RAPPEL_MINIMAL):
    proba_defauts = np.sort(proba[y == 1])[::-1]
    return proba_defauts[int(np.ceil(rappel * len(proba_defauts))) - 1]

# Seuils : sur toutes les probabilités hors pli du train (point de fonctionnement du scénario) ; la cible ne sert qu'à placer ce seuil, comme dans le scénario
SEUILS = {m: seuil_pour_rappel(probas["dpnm"].to_numpy(), probas[m].to_numpy()) for m in MODELES}
p_analyse = probas.loc[analyse["ID"]]
signale_par = pd.DataFrame({m: (p_analyse[m] >= SEUILS[m]).to_numpy() for m in MODELES})
analyse["nb_modeles_qui_signalent"] = signale_par.sum(axis=1).to_numpy()
analyse["proba_moyenne"] = p_analyse[MODELES].mean(axis=1).to_numpy()
signale = analyse["nb_modeles_qui_signalent"] > 0
analyse["groupe"] = np.select(
    [(analyse["dpnm"] == 1) & ~signale, (analyse["dpnm"] == 1) & signale, (analyse["dpnm"] == 0) & signale],
    ["Défauts manqués", "Défauts détectés", "Fausses alertes"], default="Bons clients non signalés")
ORDRE = ["Défauts manqués", "Défauts détectés", "Fausses alertes", "Bons clients non signalés"]

effectifs = analyse["groupe"].value_counts().reindex(ORDRE)
display(Markdown(f"**Modèles de `ml_{BASE}`** : {', '.join(f'{m} (seuil {s:.3f})' for m, s in SEUILS.items())}"))
display(pd.DataFrame({"Clients": effectifs, "Part de la partie d'analyse (%)": (effectifs / len(analyse) * 100).round(1),
                      "Part des défauts (%)": (effectifs / analyse["dpnm"].sum() * 100).where(effectifs.index.str.startswith("Défauts")).round(1)})
        .rename_axis("Groupe"))
display(Markdown("**Défauts : combien de modèles les signalent** (0 = manqué par tous)"))
display(pd.crosstab(analyse.loc[analyse["dpnm"] == 1, "nb_modeles_qui_signalent"], "Défauts").T)

**Modèles de `ml_14`** : CatBoost (seuil 0.492), RandomForest (seuil 0.492), LogisticRegression (seuil 0.479), MLPClassifier (seuil 0.154)

,Clients,Part de la partie d'analyse (%),Part des défauts (%)
Groupe,,,
Défauts manqués,248,5.1,32.2
Défauts détectés,523,10.8,67.8
Fausses alertes,1575,32.5,NaN
Bons clients non signalés,2493,51.5,NaN


**Défauts : combien de modèles les signalent** (0 = manqué par tous)

nb_modeles_qui_signalent,0,1,2,3,4
col_0,,,,,
Défauts,248,32,39,42,410


## 3. Profil de chaque groupe

Les indicateurs sont ceux de l'EDA et des colonnes construites (définitions : `docs/colonnes_creees.md`). Convention temporelle : `PAY_AMT1` est le paiement de septembre, qui règle la facture d'août `BILL_AMT2` ; `ratio_PAY_BILL1` = `PAY_AMT1` / `BILL_AMT2`.
- **Montants et ratios** : médianes par groupe ;
- **Codifications et catégories** : part des clients de chaque groupe.

In [4]:
NUMERIQUES = {
    "Plafond (LIMIT_BAL)": "LIMIT_BAL", "Âge": "AGE",
    "Facture de septembre (BILL_AMT1)": "BILL_AMT1", "Paiement de septembre (PAY_AMT1)": "PAY_AMT1",
    "Part de la facture d'août payée en septembre (ratio_PAY_BILL1, %)": "ratio_PAY_BILL1",
    "Part du dû de la période remboursée (ratio_PAY_BILL_global, %)": "ratio_PAY_BILL_global",
    "Utilisation du plafond en septembre (ratio_BILL_LIMIT1, %)": "ratio_BILL_LIMIT1",
    "Mois en retard sur 6 (CUMUL_INCIDENT)": "CUMUL_INCIDENT",
    "Probabilité moyenne des modèles": "proba_moyenne",
}
mediane = analyse.groupby("groupe")[list(NUMERIQUES.values())].median().reindex(ORDRE).T
mediane.index = list(NUMERIQUES)
display(Markdown("**Montants et ratios : médiane par groupe**"))
display(mediane.round(3))

def parts(colonne, libelle):
    t = pd.crosstab(analyse[colonne], analyse["groupe"], normalize="columns").reindex(columns=ORDRE) * 100
    display(Markdown(f"**{libelle} : part des clients de chaque groupe (%)**"))
    display(t.round(1))

parts("PAY_1", "Codification de septembre (PAY_1)")
parts("PAY_habituel", "Codification habituelle (PAY_habituel)")
parts("TYPE_USAGE", "Type d'usage de la carte (TYPE_USAGE)")
analyse["Aucun paiement en septembre"] = (analyse["PAY_AMT1"] == 0).map({True: "oui", False: "non"})
parts("Aucun paiement en septembre", "Aucun paiement en septembre (PAY_AMT1 = 0)")
parts("FLAG_OUVERTURE", "Compte ouvert pendant la période (FLAG_OUVERTURE)")

**Montants et ratios : médiane par groupe**

groupe,Défauts manqués,Défauts détectés,Fausses alertes,Bons clients non signalés
Plafond (LIMIT_BAL),150000.000,60000.000,80000.000,200000.000
Âge,34.500,34.000,33.000,34.000
Facture de septembre (BILL_AMT1),47941.000,13112.000,11391.000,48533.000
Paiement de septembre (PAY_AMT1),3360.500,1280.000,1379.000,4100.000
"Part de la facture d'août payée en septembre (ratio_PAY_BILL1, %)",5.515,10.141,16.554,6.819
"Part du dû de la période remboursée (ratio_PAY_BILL_global, %)",26.655,33.521,45.232,29.274
"Utilisation du plafond en septembre (ratio_BILL_LIMIT1, %)",66.572,37.406,21.796,41.088
Mois en retard sur 6 (CUMUL_INCIDENT),0.000,1.000,0.000,0.000
Probabilité moyenne des modèles,0.310,0.540,0.459,0.289


**Codification de septembre (PAY_1) : part des clients de chaque groupe (%)**

groupe,Défauts manqués,Défauts détectés,Fausses alertes,Bons clients non signalés
PAY_1,,,,
-2,6.0,3.6,6.2,9.3
-1,19.4,26.2,30.7,18.1
0,74.6,37.5,48.2,72.7
1,0.0,30.0,14.2,0.0
2,0.0,2.7,0.8,0.0


**Codification habituelle (PAY_habituel) : part des clients de chaque groupe (%)**

...

groupe,Défauts manqués,Défauts détectés,Fausses alertes,Bons clients non signalés
TYPE_USAGE,,,,
Client en difficulté,0.8,3.8,2.2,1.2
Crédit lent,53.2,31.0,26.3,47.8
Crédit rapide,12.9,21.8,20.4,15.1
Ne paie rien,0.0,7.1,2.8,0.0
Paiement comptant,22.6,23.7,33.3,25.4
Première facture,0.0,2.7,5.0,0.1
Usage mixte,10.5,9.9,10.0,10.4


**Aucun paiement en septembre (PAY_AMT1 = 0) : part des clients de chaque groupe (%)**

groupe,Défauts manqués,Défauts détectés,Fausses alertes,Bons clients non signalés
Aucun paiement en septembre,,,,
non,99.2,71.5,77.7,98.2
oui,0.8,28.5,22.3,1.8


**Compte ouvert pendant la période (FLAG_OUVERTURE) : part des clients de chaque groupe (%)**

groupe,Défauts manqués,Défauts détectés,Fausses alertes,Bons clients non signalés
FLAG_OUVERTURE,,,,
0,98.0,87.2,82.2,97.1
1,2.0,12.8,17.8,2.9


## 4. Les défauts manqués ressemblent-ils à des bons clients ?

Pour chaque indicateur numérique, l'**écart standardisé** (différence des moyennes divisée par l'écart-type commun des deux groupes) entre les défauts manqués et deux groupes de comparaison : les **bons clients non signalés** et les **défauts détectés**. Un écart proche de 0 veut dire que les deux groupes se ressemblent sur cet indicateur ; au-delà de 0,5 en valeur absolue, l'écart est net. La dernière colonne dit de quel groupe les défauts manqués sont le plus proches.

In [5]:
def ecart_standardise(a, b):
    a, b = a.dropna(), b.dropna()
    ecart_type = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    return (a.mean() - b.mean()) / ecart_type if ecart_type > 0 else 0.0

manques = analyse[analyse["groupe"] == "Défauts manqués"]
bons = analyse[analyse["groupe"] == "Bons clients non signalés"]
detectes = analyse[analyse["groupe"] == "Défauts détectés"]
lignes = []
for libelle, colonne in NUMERIQUES.items():
    e_bons, e_detectes = ecart_standardise(manques[colonne], bons[colonne]), ecart_standardise(manques[colonne], detectes[colonne])
    lignes.append({"Indicateur": libelle, "Écart avec les bons clients non signalés": round(e_bons, 2),
                   "Écart avec les défauts détectés": round(e_detectes, 2),
                   "Plus proches de": "bons clients" if abs(e_bons) < abs(e_detectes) else "défauts détectés"})
ressemblance = pd.DataFrame(lignes).set_index("Indicateur")
display(ressemblance)
display(Markdown(f"**Bilan** : les défauts manqués sont plus proches des bons clients non signalés sur "
                 f"**{(ressemblance['Plus proches de'] == 'bons clients').sum()} indicateurs sur {len(ressemblance)}**."))

,Écart avec les bons clients non signalés,Écart avec les défauts détectés,Plus proches de
Indicateur,,,
Plafond (LIMIT_BAL),-0.24,0.74,bons clients
Âge,-0.01,0.07,bons clients
Facture de septembre (BILL_AMT1),0.09,0.70,bons clients
Paiement de septembre (PAY_AMT1),-0.02,0.32,bons clients
"Part de la facture d'août payée en septembre (ratio_PAY_BILL1, %)",-0.03,-0.02,défauts détectés
"Part du dû de la période remboursée (ratio_PAY_BILL_global, %)",-0.04,-0.06,bons clients
"Utilisation du plafond en septembre (ratio_BILL_LIMIT1, %)",0.27,0.38,bons clients
Mois en retard sur 6 (CUMUL_INCIDENT),-0.06,-1.34,bons clients
Probabilité moyenne des modèles,0.38,-2.60,bons clients


**Bilan** : les défauts manqués sont plus proches des bons clients non signalés sur **8 indicateurs sur 9**.

**Comment lire** :
- **Si les défauts manqués ressemblent aux bons clients non signalés** sur la plupart des indicateurs (paiements réguliers, peu ou pas de retard, faible utilisation du plafond), rien dans les données ne les distingue : leur défaut n'était pas annoncé par leur comportement des six mois. C'est l'**issue 1** : une part de la cible est imprévisible avec ces données (définition de `dpnm` non documentée), limite à documenter dans la méthodologie et sur Streamlit.
- **S'ils forment un profil à part** (par exemple un comportement de paiement qui se dégrade, mais pas encore en retard), c'est l'**issue 2** : une piste de colonne, à justifier par la logique métier avant tout calcul, puis à juger sur les 75 % de validation.
- La conclusion s'écrit après lecture, sans chiffres recopiés à la main : elle renvoie aux tableaux.

## 5. Conclusion : une part des défauts ne s'annonce pas dans les données

**Issue 1 du protocole : une part de la cible est imprévisible avec ces données.**

- **Environ un tiers des défauts de la partie d'analyse ne sont signalés par aucun modèle** (section 2), alors que la plupart des défauts détectés le sont par les quatre modèles à la fois : les modèles s'accordent sur qui est repérable et qui ne l'est pas.
- **Ces défauts manqués ressemblent aux bons clients non signalés, pas aux défauts détectés** (sections 3 et 4) : pas de retard sur les six mois, la codification de septembre et la codification habituelle des bons clients, un paiement en septembre presque toujours, un plafond, des factures, des paiements et une part remboursée au niveau des bons clients. Les écarts standardisés avec les bons clients sont presque nuls sur la plupart des indicateurs, et les modèles leur donnent la même probabilité qu'aux bons clients non signalés.
- **Leur défaut n'était donc pas annoncé par leur comportement des six mois.** Avec ces données, ils sont indiscernables des bons clients : pour les attraper, il faudrait signaler les bons clients au même rythme, ce qui revient à un tirage au hasard. C'est une limite des données, pas des modèles, cohérente avec le plateau des scénarios et avec la définition de `dpnm`, qui n'est pas documentée.
- **Pas d'issue 2** : seul un signal faible apparaît (une utilisation du plafond un peu plus haute et un « crédit lent » un peu plus fréquent que chez les bons clients), avec un écart trop petit pour justifier une nouvelle colonne. Aucune colonne n'est construite à partir de cette analyse ; la partie de validation (75 %) n'a donc pas à être utilisée.

**Nuance** : la part des défauts manqués dépend du point de fonctionnement (60 % des défauts détectés par chaque modèle). À un rappel plus élevé, une partie d'entre eux serait signalée, mais avec à peu près autant de bons clients que de défauts.

**Suite** : cette limite est à documenter dans la méthodologie (`methodologie_ML.md`) et à présenter sur Streamlit ; la dernière étape de la phase finale est la lecture du test, une seule fois.